## **Création de la base des dépenses de l’assurance maladie**

### **Objectif**
Ce notebook vise à construire une base consolidée des dépenses d’assurance maladie sur la période **2015–2025**, en :

- agrégeant les données mensuelles issues des fichiers sources ;
- structurant les données en **séries temporelles** ;
- transformant les **catégories de dépenses en features** (colonnes) selon la **date d’observation** ;
- produisant deux sorties :
1. une base **features communes** (catégories présentes sur toute la période) ;
2. une base **union des features** (toutes catégories observées, avec valeurs manquantes si non disponibles selon les années).

In [13]:
from pathlib import Path
from datetime import datetime
import pandas as pd

MONTHS_FR = {
    "janvier": 1, "fevrier": 2, "février": 2, "mars": 3, "avril": 4, "mai": 5,
    "juin": 6, "juillet": 7, "aout": 8, "août": 8, "septembre": 9,
    "octobre": 10, "novembre": 11, "decembre": 12, "décembre": 12,
}

def _normalize(s: str) -> str:
    return (
        str(s).strip().lower()
        .replace("é", "e").replace("è", "e").replace("ê", "e")
        .replace("à", "a").replace("ù", "u").replace("û", "u")
        .replace("î", "i").replace("ï", "i").replace("ô", "o")
    )

def read_depenses_xls(path: Path) -> pd.DataFrame:
    raw = pd.read_excel(path, engine="xlrd", header=None)

    header_row = next(
        i for i in range(min(30, len(raw)))
        if pd.isna(raw.iloc[i, 0]) and raw.iloc[i, 1:].notna().sum() >= 8
    )

    header = list(raw.iloc[header_row])
    data = raw.iloc[header_row + 1 :].copy()
    data.columns = header
    data = data.rename(columns={data.columns[0]: "categorie"})

    data = data[data["categorie"].notna()].copy()
    data["categorie"] = data["categorie"].astype(str).str.strip()
    data = data[~data["categorie"].str.lower().str.startswith(("(a)", "(b)", "champ", "source"))]

    year = int(path.name[:4])
    month_cols = []
    col_to_date = {}

    for c in data.columns:
        if c == "categorie":
            continue

        if isinstance(c, (pd.Timestamp, datetime)):
            month_cols.append(c)
            col_to_date[c] = pd.Timestamp(c).to_period("M").to_timestamp()
            continue

        ns = _normalize(c)
        if ns in MONTHS_FR:
            month_cols.append(c)
            col_to_date[c] = pd.Timestamp(year=year, month=MONTHS_FR[ns], day=1)

    long_df = data.melt(
        id_vars=["categorie"],
        value_vars=month_cols,
        var_name="periode_src",
        value_name="depense",
    )

    long_df["date"] = pd.to_datetime(long_df["periode_src"].map(col_to_date), errors="coerce")
    long_df["depense"] = pd.to_numeric(long_df["depense"], errors="coerce")
    long_df = long_df.dropna(subset=["date"]).copy()
    long_df["annee"] = long_df["date"].dt.year.astype(int)

    return long_df[["categorie", "date", "annee", "depense"]]

def build_long(raw_dir: Path) -> pd.DataFrame:
    files = sorted(raw_dir.glob("*.xls"))
    if not files:
        raise FileNotFoundError(f"Aucun fichier .xls dans {raw_dir}")
    df = pd.concat([read_depenses_xls(f) for f in files], ignore_index=True)
    return df.sort_values(["date", "categorie"]).reset_index(drop=True)

# --- Exécution ---
raw_dir = (Path.cwd() / "../../data/raw/depenses-en-date-de-soins").resolve()
silver_dir = (Path.cwd() / "../../data/silver").resolve()
silver_dir.mkdir(parents=True, exist_ok=True)

df = build_long(raw_dir)

# UNION: toutes les catégories observées (NaN quand absent)
wide_union = (
    df.pivot_table(index="date", columns="categorie", values="depense", aggfunc="mean")
      .sort_index()
)
wide_union.to_csv(
    silver_dir / "depenses_timeseries_features_union_2015_2025.csv",
    sep=";",
    encoding="utf-8-sig"
)

# FEATURES (commun): seulement catégories présentes chaque année
cats_by_year = {
    y: set(g.loc[g["depense"].notna(), "categorie"].unique())
    for y, g in df.groupby("annee")
}
cats_common = sorted(set.intersection(*cats_by_year.values()))
wide_common = (
    df[df["categorie"].isin(cats_common)]
    .pivot_table(index="date", columns="categorie", values="depense", aggfunc="mean")
    .sort_index()
)
wide_common.to_csv(
    silver_dir / "depenses_timeseries_features_2015_2025.csv",
    sep=";",
    encoding="utf-8-sig"
)

print("UNION   :", wide_union.shape, "-> depenses_timeseries_features_union_2015_2025.csv")
print("FEATURES:", wide_common.shape, "-> depenses_timeseries_features_2015_2025.csv")


UNION   : (130, 30) -> depenses_timeseries_features_union_2015_2025.csv
FEATURES: (130, 21) -> depenses_timeseries_features_2015_2025.csv
